# Agent Executor v1 vs v1_icl Key/Performance Analysis

- Target runs: `round5_mrr_selector_accum_meanscore_global`, `RCT=10`
- Compares `agent_executor_v1` vs `agent_executor_v1_icl` per subset
- Reports key distribution, key-count stats, and nDCG@10 summary


In [1]:
import json
import pickle
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from json_repair import repair_json
except Exception:
    repair_json = None

BASE_RESULTS_DIR_CANDIDATES = [
    Path("../results/BRIGHT"),
    Path("results/BRIGHT"),
    Path("/data4/jongho/lattice/results/BRIGHT"),
]
RUN_TAG = "round5_mrr_selector_accum_meanscore_global"
RCT_TAG = "RCT=10"
TARGET_PROMPTS = ["agent_executor_v1", "agent_executor_v1_icl"]
CANONICAL_KEYS = {"Theory", "Entity", "Example", "Other", "theory", "entity", "example", "other"}

BASE_RESULTS_DIR = None
for cand in BASE_RESULTS_DIR_CANDIDATES:
    if cand.exists():
        BASE_RESULTS_DIR = cand
        break
if BASE_RESULTS_DIR is None:
    raise FileNotFoundError(f"No results dir found. checked={BASE_RESULTS_DIR_CANDIDATES}")

print(f"BASE_RESULTS_DIR={BASE_RESULTS_DIR}")


def extract_json_text(text: str) -> str:
    t = str(text or "").strip()
    if "```" in t:
        parts = t.split("```")
        fenced = [parts[i] for i in range(1, len(parts), 2)]
        if fenced:
            t = fenced[-1].strip()
            if t.lower().startswith("json"):
                t = t[4:].strip()
    if "{" in t and "}" in t:
        t = t[t.find("{") : t.rfind("}") + 1]
    return t


def parse_response_to_obj(response_text: str):
    t = extract_json_text(response_text)
    try:
        return json.loads(t)
    except Exception:
        if repair_json is not None:
            try:
                return repair_json(t, return_objects=True)
            except Exception:
                return None
        return None


def find_docs_obj(obj):
    if isinstance(obj, dict):
        if "Possible_Answer_Docs" in obj:
            return obj["Possible_Answer_Docs"]
        if "possible_answer_docs" in obj:
            return obj["possible_answer_docs"]
        for v in obj.values():
            found = find_docs_obj(v)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for it in obj:
            found = find_docs_obj(it)
            if found is not None:
                return found
    return None


def infer_subset_from_path(path: Path) -> str:
    parts = list(path.parts)
    if "BRIGHT" in parts:
        idx = parts.index("BRIGHT")
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return "unknown"


def infer_prompt_from_path(path: Path):
    s = str(path).replace("\\", "/")
    if "RPN=agent_executor_v1_icl" in s:
        return "agent_executor_v1_icl"
    if "RPN=agent_executor_v1-RM=concat-RE=1" in s:
        return "agent_executor_v1"
    return None


def collect_target_histories(base_dir: Path):
    picked = {}
    for p in base_dir.rglob("llm_api_history.pkl"):
        s = str(p)
        if "/round5/" not in s.replace("\\", "/"):
            continue
        if RUN_TAG not in s:
            continue
        if RCT_TAG not in s:
            continue
        prompt = infer_prompt_from_path(p)
        if prompt not in TARGET_PROMPTS:
            continue
        subset = infer_subset_from_path(p)
        key = (subset, prompt)

        # Intent: when duplicated runs exist, use the newest artifact for stable comparison.
        if key not in picked or p.stat().st_mtime > picked[key].stat().st_mtime:
            picked[key] = p
    return picked


def ndcg_from_metrics(metrics_path: Path):
    if not metrics_path.exists():
        return np.nan, np.nan
    df = pickle.load(open(metrics_path, "rb"))
    if not isinstance(df, pd.DataFrame):
        return np.nan, np.nan

    ndcg_cols = [c for c in df.columns if isinstance(c, tuple) and len(c) == 2 and c[1] == "nDCG@10"]
    if not ndcg_cols:
        return np.nan, np.nan

    ndcg_df = df[ndcg_cols].astype(float)
    ndcg_best_mean = float(ndcg_df.max(axis=1).mean())
    ndcg_last_mean = float(ndcg_df.iloc[:, -1].mean())
    return ndcg_best_mean, ndcg_last_mean


def analyze_history(history_path: Path):
    rows = pickle.load(open(history_path, "rb"))

    parsed = 0
    with_docs = 0
    parse_fail = 0
    docs_not_dict = 0
    nonstandard_resp = 0

    key_counter = Counter()
    keyset_counter = Counter()
    key_count_list = []

    for row in rows:
        obj = parse_response_to_obj(row.get("response", ""))
        if obj is None:
            parse_fail += 1
            continue
        parsed += 1

        docs = find_docs_obj(obj)
        if docs is None:
            continue
        with_docs += 1

        if not isinstance(docs, dict):
            docs_not_dict += 1
            continue

        keys = [str(k).strip() for k in docs.keys() if str(k).strip()]
        key_count_list.append(len(keys))
        keyset_counter[tuple(sorted(keys))] += 1

        has_nonstandard = False
        for k in keys:
            key_counter[k] += 1
            if k not in CANONICAL_KEYS:
                has_nonstandard = True
        if has_nonstandard:
            nonstandard_resp += 1

    metrics_path = history_path.with_name("all_eval_metrics.pkl")
    ndcg_best_mean, ndcg_last_mean = ndcg_from_metrics(metrics_path)

    key_count_arr = np.array(key_count_list, dtype=float) if key_count_list else np.array([], dtype=float)

    summary = {
        "subset": infer_subset_from_path(history_path),
        "prompt": infer_prompt_from_path(history_path),
        "history_path": str(history_path),
        "n_rows": len(rows),
        "n_parsed_json": parsed,
        "n_parse_fail": parse_fail,
        "n_with_possible_answer_docs": with_docs,
        "n_docs_not_dict": docs_not_dict,
        "unique_key_count": len(key_counter),
        "mean_keys_per_response": float(key_count_arr.mean()) if len(key_count_arr) else np.nan,
        "median_keys_per_response": float(np.median(key_count_arr)) if len(key_count_arr) else np.nan,
        "p90_keys_per_response": float(np.percentile(key_count_arr, 90)) if len(key_count_arr) else np.nan,
        "nonstandard_response_ratio": (nonstandard_resp / len(key_count_list)) if key_count_list else np.nan,
        "ndcg_best_mean": ndcg_best_mean,
        "ndcg_last_mean": ndcg_last_mean,
    }

    key_rows = []
    total_key_occ = sum(key_counter.values())
    for k, cnt in key_counter.items():
        key_rows.append(
            {
                "subset": summary["subset"],
                "prompt": summary["prompt"],
                "key": k,
                "count": cnt,
                "ratio_in_all_keys": (cnt / total_key_occ) if total_key_occ else np.nan,
                "is_nonstandard": k not in CANONICAL_KEYS,
            }
        )

    top_keysets = [
        {"key_set": " | ".join(kset), "count": cnt}
        for kset, cnt in keyset_counter.most_common(10)
    ]

    return summary, key_rows, top_keysets


picked = collect_target_histories(BASE_RESULTS_DIR)
print(f"Picked runs: {len(picked)}")
for (subset, prompt), path in sorted(picked.items()):
    print(f"- {subset:16s} | {prompt:22s} | {path}")

summary_rows = []
key_rows_all = []
keyset_map = {}
for (subset, prompt), hpath in sorted(picked.items()):
    srow, krows, top_keysets = analyze_history(hpath)
    summary_rows.append(srow)
    key_rows_all.extend(krows)
    keyset_map[(subset, prompt)] = top_keysets

df_summary = pd.DataFrame(summary_rows)
if not df_summary.empty:
    df_summary = df_summary.sort_values(["subset", "prompt"]).reset_index(drop=True)

df_keys = pd.DataFrame(key_rows_all)
if not df_keys.empty:
    df_keys = df_keys.sort_values(["subset", "prompt", "count"], ascending=[True, True, False]).reset_index(drop=True)

print()
print("[Summary per subset/prompt]")
display(df_summary)

print()
print("[Top keys per subset/prompt] (top 15)")
if len(df_keys):
    display(df_keys.groupby(["subset", "prompt"], as_index=False).head(15))
else:
    print("No key rows.")




BASE_RESULTS_DIR=../results/BRIGHT
Picked runs: 15
- biology          | agent_executor_v1      | ../results/BRIGHT/biology/round5/S=biology-TV=bottom-up-TPV=5-RInTP=/1-NumLC=10-PlTau=5.0-DC=True-RCF=0.5/LlmApiB=vllm-Llm=Qwen3-4B-Instruct-2507/NumI=10-NumES=1000-MaxBS=10-S=round5_mrr_selector_accum_meanscore_global-FT=1000/GBT=10-PreFRS=branch-RPN=agent_executor_v1-RM=concat-RE=1/RCT=10-RCS=mixed-RGT=10-RSM=meanscore_global-RRrfK=60/RRC=leaf-REM=replace/llm_api_history.pkl
- biology          | agent_executor_v1_icl  | ../results/BRIGHT/biology/round5/S=biology-TV=bottom-up-TPV=5-RInTP=/1-NumLC=10-PlTau=5.0-DC=True-RCF=0.5/LlmApiB=vllm-Llm=Qwen3-4B-Instruct-2507/NumI=10-NumES=1000-MaxBS=10-S=round5_mrr_selector_accum_meanscore_global-FT=1000/GBT=10-PreFRS=branch-RPN=agent_executor_v1_icl2-RM=concat-RE=1/RCT=10-RCS=mixed-RGT=10-RSM=meanscore_global-RRrfK=60/RRC=leaf-REM=replace/llm_api_history.pkl
- earth_science    | agent_executor_v1      | ../results/BRIGHT/earth_science/round5/S=earth

,subset,prompt,history_path,n_rows,n_parsed_json,n_parse_fail,n_with_possible_answer_docs,n_docs_not_dict,unique_key_count,mean_keys_per_response,median_keys_per_response,p90_keys_per_response,nonstandard_response_ratio,ndcg_best_mean,ndcg_last_mean
0,biology,agent_executor_v1,../results/BRIGHT/biology/round5/S=biology-TV=...,869,869,0,869,0,4,4.000000,4.0,4.0,0.000000,76.335665,64.724433
1,biology,agent_executor_v1_icl,../results/BRIGHT/biology/round5/S=biology-TV=...,840,840,0,840,0,112,4.172619,4.0,5.0,0.998810,73.558215,61.706541
2,earth_science,agent_executor_v1,../results/BRIGHT/earth_science/round5/S=earth...,894,894,0,883,0,4,4.000000,4.0,4.0,0.000000,70.549916,58.139068
3,earth_science,agent_executor_v1_icl,../results/BRIGHT/earth_science/round5/S=earth...,901,901,0,901,0,130,4.123196,4.0,5.0,0.998890,72.711739,59.723334
4,economics,agent_executor_v1,../results/BRIGHT/economics/round5/S=economics...,865,865,0,865,0,4,4.000000,4.0,4.0,0.000000,40.033252,28.507906
5,economics,agent_executor_v1_icl,../results/BRIGHT/economics/round5/S=economics...,825,825,0,825,0,134,4.184242,4.0,5.0,0.997576,40.405792,28.758532
6,pony,agent_executor_v1_icl,../results/BRIGHT/pony/round5/S=pony-TV=bottom...,1066,1066,0,1052,0,92,4.069392,4.0,4.0,1.000000,20.164059,12.568087
7,psychology,agent_executor_v1,../results/BRIGHT/psychology/round5/S=psycholo...,553,553,0,553,0,4,4.000000,4.0,4.0,0.000000,56.689152,45.962705
8,psychology,agent_executor_v1_icl,../results/BRIGHT/psychology/round5/S=psycholo...,769,769,0,769,0,168,4.287386,4.0,5.0,1.000000,56.638460,44.435676
9,robotics,agent_executor_v1,../results/BRIGHT/robotics/round5/S=robotics-T...,911,911,0,908,0,4,4.000000,4.0,4.0,0.000000,42.894445,26.679289



[Top keys per subset/prompt] (top 15)


,subset,prompt,key,count,ratio_in_all_keys,is_nonstandard
0,biology,agent_executor_v1,Theory,869,0.250000,False
1,biology,agent_executor_v1,Entity,869,0.250000,False
2,biology,agent_executor_v1,Example,869,0.250000,False
3,biology,agent_executor_v1,Other,869,0.250000,False
4,biology,agent_executor_v1_icl,Theory,840,0.239658,False
...,...,...,...,...,...,...
1240,theoremqa_theorems,agent_executor_v1_icl,Contrast,14,0.004580,True
1241,theoremqa_theorems,agent_executor_v1_icl,Alternative,13,0.004253,True
1242,theoremqa_theorems,agent_executor_v1_icl,Generalization,12,0.003925,True
1243,theoremqa_theorems,agent_executor_v1_icl,Method,10,0.003271,True


In [2]:
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# Subset-wise v1 vs v1_icl comparison table

if df_summary.empty:
    print("No runs found for the configured filters.")
else:
    perf_cols = ["ndcg_best_mean", "ndcg_last_mean"]
    key_stat_cols = [
        "unique_key_count",
        "mean_keys_per_response",
        "median_keys_per_response",
        "p90_keys_per_response",
        "nonstandard_response_ratio",
    ]

    wide_parts = []
    for col in perf_cols + key_stat_cols:
        pvt = df_summary.pivot(index="subset", columns="prompt", values=col)
        pvt.columns = [f"{col}__{c}" for c in pvt.columns]
        wide_parts.append(pvt)

    comp = pd.concat(wide_parts, axis=1).reset_index()

    # Intent: explicit delta columns make v1_icl degradation sources easy to inspect.
    for col in perf_cols + key_stat_cols:
        a = f"{col}__agent_executor_v1_icl"
        b = f"{col}__agent_executor_v1"
        if a in comp.columns and b in comp.columns:
            comp[f"delta({col})_icl_minus_v1"] = comp[a] - comp[b]

    print("[v1 vs v1_icl: per-subset comparison]")
    display(comp.sort_values("subset"))

    print()
    print("[Non-standard keys only: v1_icl]")
    if not df_keys.empty:
        nonstd = df_keys[(df_keys["prompt"] == "agent_executor_v1_icl") & (df_keys["is_nonstandard"])].copy()
        nonstd = nonstd.sort_values(["subset", "count"], ascending=[True, False])
        display(nonstd.groupby("subset", as_index=False).head(15))
    else:
        print("No key rows.")

    print()
    print("[Top key-set patterns per subset/prompt] (top 5)")
    rows = []
    for (subset, prompt), items in keyset_map.items():
        for it in items[:5]:
            rows.append(
                {
                    "subset": subset,
                    "prompt": prompt,
                    "key_set": it["key_set"],
                    "count": it["count"],
                }
            )
    if rows:
        display(pd.DataFrame(rows).sort_values(["subset", "prompt", "count"], ascending=[True, True, False]))
    else:
        print("No key-set rows.")




[v1 vs v1_icl: per-subset comparison]


,subset,ndcg_best_mean__agent_executor_v1,ndcg_best_mean__agent_executor_v1_icl,ndcg_last_mean__agent_executor_v1,ndcg_last_mean__agent_executor_v1_icl,unique_key_count__agent_executor_v1,unique_key_count__agent_executor_v1_icl,mean_keys_per_response__agent_executor_v1,mean_keys_per_response__agent_executor_v1_icl,median_keys_per_response__agent_executor_v1,...,p90_keys_per_response__agent_executor_v1_icl,nonstandard_response_ratio__agent_executor_v1,nonstandard_response_ratio__agent_executor_v1_icl,delta(ndcg_best_mean)_icl_minus_v1,delta(ndcg_last_mean)_icl_minus_v1,delta(unique_key_count)_icl_minus_v1,delta(mean_keys_per_response)_icl_minus_v1,delta(median_keys_per_response)_icl_minus_v1,delta(p90_keys_per_response)_icl_minus_v1,delta(nonstandard_response_ratio)_icl_minus_v1
0,biology,76.335665,73.558215,64.724433,61.706541,4.0,112.0,4.0,4.172619,4.0,...,5.0,0.0,0.998810,-2.777450,-3.017892,108.0,0.172619,0.0,1.0,0.998810
1,earth_science,70.549916,72.711739,58.139068,59.723334,4.0,130.0,4.0,4.123196,4.0,...,5.0,0.0,0.998890,2.161822,1.584266,126.0,0.123196,0.0,1.0,0.998890
2,economics,40.033252,40.405792,28.507906,28.758532,4.0,134.0,4.0,4.184242,4.0,...,5.0,0.0,0.997576,0.372540,0.250626,130.0,0.184242,0.0,1.0,0.997576
3,pony,NaN,20.164059,NaN,12.568087,NaN,92.0,NaN,4.069392,NaN,...,4.0,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,psychology,56.689152,56.638460,45.962705,44.435676,4.0,168.0,4.0,4.287386,4.0,...,5.0,0.0,1.000000,-0.050692,-1.527029,164.0,0.287386,0.0,1.0,1.000000
5,robotics,42.894445,42.830556,26.679289,26.340960,4.0,83.0,4.0,4.083333,4.0,...,4.0,0.0,1.000000,-0.063889,-0.338329,79.0,0.083333,0.0,0.0,1.000000
6,stackoverflow,NaN,33.821222,NaN,27.445903,NaN,84.0,NaN,4.038144,NaN,...,4.0,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,sustainable_living,NaN,44.630454,NaN,34.818870,NaN,127.0,NaN,4.153598,NaN,...,5.0,NaN,0.998926,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,theoremqa_questions,NaN,25.098492,NaN,19.991571,NaN,280.0,NaN,4.198491,NaN,...,5.0,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,theoremqa_theorems,NaN,50.420190,NaN,41.376852,NaN,180.0,NaN,4.462774,NaN,...,5.0,NaN,0.998540,NaN,NaN,NaN,NaN,NaN,NaN,NaN



[Non-standard keys only: v1_icl]


,subset,prompt,key,count,ratio_in_all_keys,is_nonstandard
7,biology,agent_executor_v1_icl,(Other),703,0.200571,True
8,biology,agent_executor_v1_icl,Mechanism,101,0.028816,True
9,biology,agent_executor_v1_icl,Analog,10,0.002853,True
10,biology,agent_executor_v1_icl,Evolutionary Advantage,6,0.001712,True
11,biology,agent_executor_v1_icl,Alternative_Mechanism,6,0.001712,True
...,...,...,...,...,...,...
1243,theoremqa_theorems,agent_executor_v1_icl,Method,10,0.003271,True
1244,theoremqa_theorems,agent_executor_v1_icl,Justification,9,0.002944,True
1245,theoremqa_theorems,agent_executor_v1_icl,Combinatorial Justification,8,0.002617,True
1246,theoremqa_theorems,agent_executor_v1_icl,Comparison,7,0.002290,True



[Top key-set patterns per subset/prompt] (top 5)


,subset,prompt,key_set,count
0,biology,agent_executor_v1,Entity | Example | Other | Theory,869
1,biology,agent_executor_v1_icl,(Other) | Entity | Example | Theory,684
2,biology,agent_executor_v1_icl,(Other) | Entity | Example | Mechanism | Theory,15
3,biology,agent_executor_v1_icl,Analog | Entity | Example | Mechanism | Theory,8
4,biology,agent_executor_v1_icl,Entity | Example | Limitation | Mechanism | Th...,5
5,biology,agent_executor_v1_icl,Entity | Example | Mechanism | Theory,5
6,earth_science,agent_executor_v1,Entity | Example | Other | Theory,883
7,earth_science,agent_executor_v1_icl,(Other) | Entity | Example | Theory,676
8,earth_science,agent_executor_v1_icl,Entity | Example | Mechanism | Theory,31
9,earth_science,agent_executor_v1_icl,(Alternative) | Entity | Example | Theory,9


In [3]:
# ICL2 key examples (appendix cell)
# Intent: provide concrete qualitative examples of key drift for agent_executor_v1_icl2.

from pathlib import Path
import pickle
import json

TARGET_SUBSETS = ["biology", "earth_science", "economics", "psychology", "robotics"]
RUN_TAG = "round5_mrr_selector_accum_meanscore_global"
RCT_TAG = "RCT=10"
ICL2_TAG = "RPN=agent_executor_v1_icl2-RM=concat-RE=1"
CANONICAL_KEYS = {"Theory", "Entity", "Example", "Other", "theory", "entity", "example", "other"}


def _extract_json_text_local(text: str) -> str:
    t = str(text or "").strip()
    if "```" in t:
        parts = t.split("```")
        fenced = [parts[i] for i in range(1, len(parts), 2)]
        if fenced:
            t = fenced[-1].strip()
            if t.lower().startswith("json"):
                t = t[4:].strip()
    if "{" in t and "}" in t:
        t = t[t.find("{") : t.rfind("}") + 1]
    return t


def _find_docs_local(obj):
    if isinstance(obj, dict):
        if "Possible_Answer_Docs" in obj:
            return obj["Possible_Answer_Docs"]
        if "possible_answer_docs" in obj:
            return obj["possible_answer_docs"]
        for v in obj.values():
            found = _find_docs_local(v)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for it in obj:
            found = _find_docs_local(it)
            if found is not None:
                return found
    return None


def _pick_icl2_histories(base_dir: Path):
    picked = {}
    for p in base_dir.rglob("llm_api_history.pkl"):
        s = str(p)
        if "/round5/" not in s.replace("\\", "/"):
            continue
        if RUN_TAG not in s or RCT_TAG not in s or ICL2_TAG not in s:
            continue
        parts = list(p.parts)
        if "BRIGHT" not in parts:
            continue
        subset = parts[parts.index("BRIGHT") + 1]
        if subset not in TARGET_SUBSETS:
            continue
        if subset not in picked or p.stat().st_mtime > picked[subset].stat().st_mtime:
            picked[subset] = p
    return picked


base_candidates = [Path("../results/BRIGHT"), Path("results/BRIGHT"), Path("/data4/jongho/lattice/results/BRIGHT")]
base_dir = None
for c in base_candidates:
    if c.exists():
        base_dir = c
        break
if base_dir is None:
    raise FileNotFoundError(f"No results dir found. checked={base_candidates}")

picked = _pick_icl2_histories(base_dir)
print("ICL2 histories found:")
for subset, p in sorted(picked.items()):
    print(f"- {subset}: {p}")

example_rows = []

for subset, hist_path in sorted(picked.items()):
    rows = pickle.load(open(hist_path, "rb"))
    shown_nonstd = 0
    shown_canonical = 0

    for row_idx, row in enumerate(rows):
        raw = row.get("response", "")
        candidate = _extract_json_text_local(raw)
        try:
            obj = json.loads(candidate)
        except Exception:
            continue
        docs = _find_docs_local(obj)
        if not isinstance(docs, dict):
            continue

        keys = [str(k).strip() for k in docs.keys() if str(k).strip()]
        nonstd_keys = [k for k in keys if k not in CANONICAL_KEYS]
        plan = str(obj.get("Plan", "") or "").strip()

        if nonstd_keys and shown_nonstd < 3:
            example_rows.append(
                {
                    "subset": subset,
                    "type": "nonstandard",
                    "row_idx": row_idx,
                    "keys": keys,
                    "nonstd_keys": nonstd_keys,
                    "plan_preview": plan[:240],
                    "response_preview": str(raw)[:360],
                }
            )
            shown_nonstd += 1
        elif (not nonstd_keys) and shown_canonical < 1:
            example_rows.append(
                {
                    "subset": subset,
                    "type": "canonical_only",
                    "row_idx": row_idx,
                    "keys": keys,
                    "nonstd_keys": [],
                    "plan_preview": plan[:240],
                    "response_preview": str(raw)[:360],
                }
            )
            shown_canonical += 1

        if shown_nonstd >= 3 and shown_canonical >= 1:
            break


ex_df = pd.DataFrame(example_rows)
if ex_df.empty:
    print("No ICL2 examples found.")
else:
    print("\n[ICL2 key examples]")
    display(ex_df[["subset", "type", "row_idx", "keys", "nonstd_keys", "plan_preview"]])

    print("\n[Detailed raw preview per subset]")
    for subset in sorted(ex_df["subset"].unique()):
        print("\n===", subset, "===")
        sub = ex_df[ex_df["subset"] == subset]
        for _, r in sub.iterrows():
            print(f"- type={r['type']}, row_idx={int(r['row_idx'])}")
            print("  keys:", r["keys"])
            if r["nonstd_keys"]:
                print("  nonstd_keys:", r["nonstd_keys"])
            print("  response_preview:", str(r["response_preview"]).replace("\n", " ")[:300])



ICL2 histories found:
- biology: ../results/BRIGHT/biology/round5/S=biology-TV=bottom-up-TPV=5-RInTP=/1-NumLC=10-PlTau=5.0-DC=True-RCF=0.5/LlmApiB=vllm-Llm=Qwen3-4B-Instruct-2507/NumI=10-NumES=1000-MaxBS=10-S=round5_mrr_selector_accum_meanscore_global-FT=1000/GBT=10-PreFRS=branch-RPN=agent_executor_v1_icl2-RM=concat-RE=1/RCT=10-RCS=mixed-RGT=10-RSM=meanscore_global-RRrfK=60/RRC=leaf-REM=replace/llm_api_history.pkl
- earth_science: ../results/BRIGHT/earth_science/round5/S=earth_science-TV=bottom-up-TPV=5-RInTP=/1-NumLC=10-PlTau=5.0-DC=True-RCF=0.5/LlmApiB=vllm-Llm=Qwen3-4B-Instruct-2507/NumI=10-NumES=1000-MaxBS=10-S=round5_mrr_selector_accum_meanscore_global-FT=1000/GBT=10-PreFRS=branch-RPN=agent_executor_v1_icl2-RM=concat-RE=1/RCT=10-RCS=mixed-RGT=10-RSM=meanscore_global-RRrfK=60/RRC=leaf-REM=replace/llm_api_history.pkl
- economics: ../results/BRIGHT/economics/round5/S=economics-TV=bottom-up-TPV=5-RInTP=/1-NumLC=10-PlTau=5.0-DC=True-RCF=0.5/LlmApiB=vllm-Llm=Qwen3-4B-Instruct-2507/NumI=

,subset,type,row_idx,keys,nonstd_keys,plan_preview
0,biology,nonstandard,0,"[Theory, Entity, Example, (Other)]",[(Other)],The user is asking whether kissing is a natura...
1,biology,nonstandard,1,"[Theory, Entity, Example, (Other)]",[(Other)],The user's core intent is to understand the sc...
2,biology,nonstandard,2,"[Theory, Entity, Example, (Other)]",[(Other)],The user observes a unilateral airflow during ...
3,biology,canonical_only,435,"[Theory, Entity, Example]",[],The user is exploring whether monogamy in huma...
4,earth_science,nonstandard,0,"[Theory, Entity, Example, (Other)]",[(Other)],The user's core intent is to understand how Ea...
5,earth_science,nonstandard,1,"[Theory, Entity, Example, (Other)]",[(Other)],The user is observing a counterintuitive pheno...
6,earth_science,nonstandard,2,"[Theory, Entity, Example, (Other)]",[(Other)],The user is seeking a clear classification of ...
7,earth_science,canonical_only,470,"[Theory, Entity, Example, Other]",[],The user is asking whether naturally occurring...
8,economics,nonstandard,0,"[Theory, Entity, Example, (Other)]",[(Other)],The user is seeking a precise term or conceptu...
9,economics,nonstandard,1,"[Theory, Entity, Example, (Other)]",[(Other)],The user is exploring whether Gaza could have ...



[Detailed raw preview per subset]

=== biology ===
- type=nonstandard, row_idx=0
  keys: ['Theory', 'Entity', 'Example', '(Other)']
  nonstd_keys: ['(Other)']
  response_preview: ```json {   "Plan": "The user is asking whether kissing is a naturally occurring, instinctive human behavior—distinct from a socially constructed one—by comparing it to animal behaviors and evolutionary origins. The core evidence would include biological and evolutionary parallels in non-human speci
- type=nonstandard, row_idx=1
  keys: ['Theory', 'Entity', 'Example', '(Other)']
  nonstd_keys: ['(Other)']
  response_preview: ```json {   "Plan": "The user's core intent is to understand the scientific mechanism behind joint cracking noise and assess its potential harm. The evidence needed includes a clear explanation of the physical process—such as cavitation in synovial fluid—and plausible alternative mechanisms or relat
- type=nonstandard, row_idx=2
  keys: ['Theory', 'Entity', 'Example', '(Other)']
  nonstd_

In [4]:
# ICL2 non-standard key frequency summary (quantitative)
# Intent: complement qualitative examples with subset-wise and global non-standard key frequency tables.

from collections import Counter, defaultdict

TOP_N = 30
TARGET_SUBSETS = ["biology", "earth_science", "economics", "psychology", "robotics"]
RUN_TAG = "round5_mrr_selector_accum_meanscore_global"
RCT_TAG = "RCT=10"
ICL2_TAG = "RPN=agent_executor_v1_icl2-RM=concat-RE=1"
CANONICAL_KEYS = {"Theory", "Entity", "Example", "Other", "theory", "entity", "example", "other"}


def _extract_json_text_local_q(text: str) -> str:
    t = str(text or "").strip()
    if "```" in t:
        parts = t.split("```")
        fenced = [parts[i] for i in range(1, len(parts), 2)]
        if fenced:
            t = fenced[-1].strip()
            if t.lower().startswith("json"):
                t = t[4:].strip()
    if "{" in t and "}" in t:
        t = t[t.find("{") : t.rfind("}") + 1]
    return t


def _find_docs_local_q(obj):
    if isinstance(obj, dict):
        if "Possible_Answer_Docs" in obj:
            return obj["Possible_Answer_Docs"]
        if "possible_answer_docs" in obj:
            return obj["possible_answer_docs"]
        for v in obj.values():
            found = _find_docs_local_q(v)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for it in obj:
            found = _find_docs_local_q(it)
            if found is not None:
                return found
    return None


def _pick_icl2_histories_q(base_dir: Path):
    picked = {}
    for p in base_dir.rglob("llm_api_history.pkl"):
        s = str(p)
        if "/round5/" not in s.replace("\\", "/"):
            continue
        if RUN_TAG not in s or RCT_TAG not in s or ICL2_TAG not in s:
            continue
        parts = list(p.parts)
        if "BRIGHT" not in parts:
            continue
        subset = parts[parts.index("BRIGHT") + 1]
        if subset not in TARGET_SUBSETS:
            continue
        if subset not in picked or p.stat().st_mtime > picked[subset].stat().st_mtime:
            picked[subset] = p
    return picked


base_candidates = [Path("../results/BRIGHT"), Path("results/BRIGHT"), Path("/data4/jongho/lattice/results/BRIGHT")]
base_dir = None
for c in base_candidates:
    if c.exists():
        base_dir = c
        break
if base_dir is None:
    raise FileNotFoundError(f"No results dir found. checked={base_candidates}")

picked = _pick_icl2_histories_q(base_dir)

subset_counter = {subset: Counter() for subset in sorted(picked.keys())}
global_counter = Counter()
subset_presence = defaultdict(set)
subset_total_responses = defaultdict(int)
subset_nonstd_responses = defaultdict(int)

for subset, hist_path in sorted(picked.items()):
    rows = pickle.load(open(hist_path, "rb"))
    for row in rows:
        raw = row.get("response", "")
        cand = _extract_json_text_local_q(raw)
        try:
            obj = json.loads(cand)
        except Exception:
            continue
        docs = _find_docs_local_q(obj)
        if not isinstance(docs, dict):
            continue

        keys = [str(k).strip() for k in docs.keys() if str(k).strip()]
        nonstd_keys = [k for k in keys if k not in CANONICAL_KEYS]

        subset_total_responses[subset] += 1
        if nonstd_keys:
            subset_nonstd_responses[subset] += 1

        for key in nonstd_keys:
            subset_counter[subset][key] += 1
            global_counter[key] += 1
            subset_presence[key].add(subset)

print("[ICL2 non-standard response ratio by subset]")
ratio_rows = []
for subset in sorted(subset_total_responses.keys()):
    total = int(subset_total_responses[subset])
    nonstd = int(subset_nonstd_responses[subset])
    ratio_rows.append(
        {
            "subset": subset,
            "responses_with_docs": total,
            "responses_with_nonstd": nonstd,
            "nonstd_ratio": (nonstd / total) if total else float("nan"),
        }
    )
display(pd.DataFrame(ratio_rows).sort_values("subset"))

print("\n[Top non-standard keys per subset]")
per_subset_rows = []
for subset in sorted(subset_counter.keys()):
    for key, cnt in subset_counter[subset].most_common(TOP_N):
        per_subset_rows.append({"subset": subset, "key": key, "count": cnt})
if per_subset_rows:
    display(pd.DataFrame(per_subset_rows).sort_values(["subset", "count"], ascending=[True, False]))
else:
    print("No non-standard keys found.")

print("\n[Global top non-standard keys]")
global_rows = []
for key, cnt in global_counter.most_common(TOP_N):
    global_rows.append(
        {
            "key": key,
            "count": cnt,
            "num_subsets_present": len(subset_presence[key]),
            "subsets": ", ".join(sorted(subset_presence[key])),
        }
    )
if global_rows:
    display(pd.DataFrame(global_rows))
else:
    print("No non-standard keys found.")



[ICL2 non-standard response ratio by subset]


,subset,responses_with_docs,responses_with_nonstd,nonstd_ratio
0,biology,840,839,0.998810
1,earth_science,900,899,0.998889
2,economics,798,796,0.997494
3,psychology,767,767,1.000000
4,robotics,924,924,1.000000



[Top non-standard keys per subset]


,subset,key,count
0,biology,(Other),703
1,biology,Mechanism,101
2,biology,Analog,10
3,biology,Evolutionary Advantage,6
4,biology,Alternative_Mechanism,6
...,...,...,...
145,robotics,Source,2
146,robotics,(System Command),2
147,robotics,Package Specification,2
148,robotics,Known_Issue,2



[Global top non-standard keys]


,key,count,num_subsets_present,subsets
0,(Other),3269,5,"biology, earth_science, economics, psychology,..."
1,Mechanism,527,5,"biology, earth_science, economics, psychology,..."
2,Model,39,4,"biology, earth_science, economics, psychology"
3,Analog,35,5,"biology, earth_science, economics, psychology,..."
4,(Alternative),28,3,"earth_science, psychology, robotics"
5,Alternative,26,5,"biology, earth_science, economics, psychology,..."
6,Framework,18,3,"economics, psychology, robotics"
7,Reference,16,3,"earth_science, economics, robotics"
8,Contrast,15,4,"biology, earth_science, economics, psychology"
9,Analogous Phenomenon,15,3,"biology, earth_science, psychology"
